In [8]:
!python -m pip install python-dotenv

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip available: 22.3 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
!python -m pip install requests

     ---------------------------------------- 64.7/64.7 kB 3.4 MB/s eta 0:00:00
     -------------------------------------- 107.1/107.1 kB 6.1 MB/s eta 0:00:00
     ---------------------------------------- 70.4/70.4 kB 4.0 MB/s eta 0:00:00
     ---------------------------------------- 129.8/129.8 kB ? eta 0:00:00
     -------------------------------------- 163.3/163.3 kB 9.6 MB/s eta 0:00:00


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip available: 22.3 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from dotenv import load_dotenv
import os
from requests import post, get
import base64
import json

In [2]:
load_dotenv()
client_id = os.getenv('CLIENT-ID')
client_secret = os.getenv('CLIENT-SECRET')

**Obtain Access Token**

In [3]:
def get_token():
    auth_string = client_id + ':' + client_secret
    auth_bytes = auth_string.encode('utf-8')
    auth_base64 = str(base64.b64encode(auth_bytes), 'utf-8')

    url = 'https://accounts.spotify.com/api/token'
    headers = {
        'Authorization': 'Basic ' + auth_base64,
        'Content-Type': 'application/x-www-form-urlencoded'
    }
    data = {'grant_type': 'client_credentials'}
    result = post(url, headers=headers, data=data)
    json_result = json.loads(result.content)
    token=json_result['access_token']
    return token

In [4]:
token=get_token()

In [5]:
def get_auth_header(token):
    return {'Authorization': 'Bearer ' + token} 

In [32]:
def search_artist(token, artist_name):
    url = 'https://api.spotify.com/v1/search'
    headers = get_auth_header(token)
    query = f'?q={artist_name}&type=artist&limit=1'
    query_url = url + query
    result = get(query_url, headers=headers)
    json_result = json.loads(result.content)['artists']['items']
    if len(json_result) == 0:
        return None
    return json_result[0]

In [33]:
search_artist(token, 'Lana Del Rey')

{'external_urls': {'spotify': 'https://open.spotify.com/artist/00FQb4jTyendYWaN8pK0wa'},
 'followers': {'href': None, 'total': 51359943},
 'genres': [],
 'href': 'https://api.spotify.com/v1/artists/00FQb4jTyendYWaN8pK0wa',
 'id': '00FQb4jTyendYWaN8pK0wa',
 'images': [{'url': 'https://i.scdn.co/image/ab6761610000e5ebb99cacf8acd5378206767261',
   'height': 640,
   'width': 640},
  {'url': 'https://i.scdn.co/image/ab67616100005174b99cacf8acd5378206767261',
   'height': 320,
   'width': 320},
  {'url': 'https://i.scdn.co/image/ab6761610000f178b99cacf8acd5378206767261',
   'height': 160,
   'width': 160}],
 'name': 'Lana Del Rey',
 'popularity': 88,
 'type': 'artist',
 'uri': 'spotify:artist:00FQb4jTyendYWaN8pK0wa'}

In [ ]:
def get__songs_by_artist(token, artist_id):
    url = f'https://api.spotify.com/v1/artists/{artist_id}/top-tracks'
    headers = get_auth_header(token)
    result = get(url, headers=headers)
    json_result = json.loads(result.content)['tracks']
    return json_result
result=search_artist(token, 'Lana Del Rey')
artist_id=result['id']
songs=get__songs_by_artist(token, artist_id)


In [35]:
for idx, song in enumerate(songs):
    print(f"{idx+1}. {song['name']} - Popularity: {song['popularity']}")

1. Young And Beautiful - Popularity: 87
2. Summertime Sadness - Popularity: 63
3. Cinnamon Girl - Popularity: 86
4. Say Yes To Heaven - Popularity: 84
5. Brooklyn Baby - Popularity: 83
6. Video Games - Popularity: 60
7. West Coast - Popularity: 82
8. Born To Die - Popularity: 59
9. Diet Mountain Dew - Popularity: 53
10. Margaret (feat. Bleachers) - Popularity: 80


**Get artist Most Stream By Genre**

In [10]:
def get_artist_by_genre(token, genre):
    url = 'https://api.spotify.com/v1/search'
    headers = get_auth_header(token)
    query = f'?q=genre:%22{genre}%22&type=artist&limit=5'
    query_url = url + query
    result = get(query_url, headers=headers)
    json_result = json.loads(result.content)['artists']['items']
    if len(json_result) == 0:
        return None
    return json_result

In [11]:
artists=get_artist_by_genre(token, 'pop')
for idx, artist in enumerate(artists):
    print(f"{idx+1}. {artist['name']} - Followers:{artist['followers']['total']} - Popularity: {artist['popularity']}")

1. Taylor Swift - Followers:144580633 - Popularity: 100
2. David Guetta - Followers:27086576 - Popularity: 86
3. Billie Eilish - Followers:118136424 - Popularity: 90
4. Rihanna - Followers:68679185 - Popularity: 89
5. Lady Gaga - Followers:41899829 - Popularity: 88


**Get top genre Music by Year**

In [36]:
def get_genre_by_song(token, song_name):
    url = 'https://api.spotify.com/v1/search'
    headers = get_auth_header(token)
    query = f'?q={song_name}&type=track&limit=1'
    query_url = url + query
    result = get(query_url, headers=headers)
    json_result = json.loads(result.content)['tracks']['items']
    if len(json_result) == 0:
        return None
    artist_id=json_result[0]['artists'][0]['id']
    url_artist=f'https://api.spotify.com/v1/artists/{artist_id}'
    result_artist = get(url_artist, headers=headers)
    json_result_artist = json.loads(result_artist.content)
    return json_result_artist['genres']

In [43]:
songs

[{'album': {'album_type': 'single',
   'artists': [{'external_urls': {'spotify': 'https://open.spotify.com/artist/00FQb4jTyendYWaN8pK0wa'},
     'href': 'https://api.spotify.com/v1/artists/00FQb4jTyendYWaN8pK0wa',
     'id': '00FQb4jTyendYWaN8pK0wa',
     'name': 'Lana Del Rey',
     'type': 'artist',
     'uri': 'spotify:artist:00FQb4jTyendYWaN8pK0wa'}],
   'external_urls': {'spotify': 'https://open.spotify.com/album/1D92WOHWUI2AGQCCdplcXL'},
   'href': 'https://api.spotify.com/v1/albums/1D92WOHWUI2AGQCCdplcXL',
   'id': '1D92WOHWUI2AGQCCdplcXL',
   'images': [{'url': 'https://i.scdn.co/image/ab67616d0000b273d7fb3e4c63020039d1cff6b2',
     'height': 640,
     'width': 640},
    {'url': 'https://i.scdn.co/image/ab67616d00001e02d7fb3e4c63020039d1cff6b2',
     'height': 300,
     'width': 300},
    {'url': 'https://i.scdn.co/image/ab67616d00004851d7fb3e4c63020039d1cff6b2',
     'height': 64,
     'width': 64}],
   'is_playable': True,
   'name': 'Young And Beautiful',
   'release_date': 

In [44]:
for idx, song in enumerate(songs):
    genre=get_genre_by_song(token, song['name'])
    print(f"Song: {song['name']} - Genre: {genre}")

Song: Young And Beautiful - Genre: []
Song: Summertime Sadness - Genre: []
Song: Cinnamon Girl - Genre: []
Song: Say Yes To Heaven - Genre: []
Song: Brooklyn Baby - Genre: []
Song: Video Games - Genre: []
Song: West Coast - Genre: []
Song: Born To Die - Genre: []
Song: Diet Mountain Dew - Genre: []
Song: Margaret (feat. Bleachers) - Genre: []


In [31]:
genre

[]

**Get all Genre of music**

In [25]:
year_genres=get_genre_by_year(token, 2010, 2020)

In [28]:
year_genres[4].keys()

dict_keys(['album', 'artists', 'available_markets', 'disc_number', 'duration_ms', 'explicit', 'external_ids', 'external_urls', 'href', 'id', 'is_local', 'is_playable', 'name', 'popularity', 'preview_url', 'track_number', 'type', 'uri'])

In [18]:
year=get_genre_by_year(token, 2010, 2020)
for idx, song in enumerate(year):
    print(f"{idx+1}. {song['name']} - Album: {song['album']['album_type']}  - Popularity: {song['popularity']} - Artist: {song['artists'][0]['name']}")

1. 7 Years - Album: compilation  - Popularity: 17 - Artist: Lukas Graham
2. We Are Young - Album: compilation  - Popularity: 37 - Artist: fun.
3. Gone (Acoustic 2010) - Album: album  - Popularity: 23 - Artist: NOTHING MORE
4. High Hopes - Album: compilation  - Popularity: 23 - Artist: Panic! At The Disco
5. King - Album: compilation  - Popularity: 14 - Artist: Olly Alexander (Years & Years)
6. Treat You Better - Album: compilation  - Popularity: 9 - Artist: Shawn Mendes
7. All Of A Sudden - Remastered 2010 - Album: compilation  - Popularity: 25 - Artist: Matt Monro
8. We Are Young (feat. Janelle Monáe) - Album: compilation  - Popularity: 6 - Artist: fun.
9. Radar Detector - Live At The BBC for Zane Lowe / 5th May, 2010 - Album: album  - Popularity: 3 - Artist: Darwin Deez
10. Up In The Clouds - Live At The BBC for Zane Lowe / 5th May, 2010 - Album: album  - Popularity: 4 - Artist: Darwin Deez
